# Multifield `solveField_new` test — Stokes system

This notebook validates the new multifield `prepare → solve → reconstruct` pipeline.
The velocity field is full order, while the pressure field uses `reducedOrder=true`,
so the test exercises a block-diagonal transformation with different field spaces.


In [1]:
using LowLevelFEM
using LinearAlgebra
using SparseArrays
using Test

solve_path = joinpath(
    dirname(pathof(LowLevelFEM)),
    "solve.jl"
)

@assert isfile(solve_path) "Copy solve.jl into the LowLevelFEM src directory first: $solve_path"

Base.include(
    LowLevelFEM,
    solve_path
)

using LowLevelFEM: solveField_new
using LowLevelFEM: prepare_multifield_system, solve_linear_system


## Model and block system

The mixed system contains a full-order velocity field `V` and a reduced-order pressure field `P`.
The existing `solveField` result is retained as a reference for this first multifield validation.


In [2]:
structured_rect_mesh(lx=2, order=3)
material = Material("body")

P = Field(
    [material],
    type=:ScalarField,
    fieldName=:p,
    rhsName=:fp,
    dim=2,
    reducedOrder=true
)

V = Field(
    [material],
    type=:VectorField,
    fieldName=:v,
    rhsName=:fv,
    dim=2
)

pres1 = BoundaryCondition("rightbottom", field=P, p=0)
suppT = BoundaryCondition("top", field=V, vx=0, vy=0)
suppB = BoundaryCondition("bottom", field=V, vx=0, vy=0)
support = [pres1, suppT, suppB]

load_v = LoadCondition("body", fvx=1.0, fvy=0.0)
fv = loadVector(V, [load_v])
gp = loadVector(P, [])
F = SystemVector([fv, gp])

μ = 1.0
γ = 1e-1

A = ∫((SymGrad(V) ⋅ SymGrad(V)) * 2μ)
B = ∫(Div(V) ⋅ P)
C = 0 * ∫(P ⋅ P)
D = ∫(Div(V) ⋅ Div(V) * γ)

K = SystemMatrix([
    A + D  B
    B'     C
])


sparse([1, 2, 9, 10, 47, 48, 49, 50, 319, 320  …  3765, 3766, 3775, 3776, 3777, 3778, 3779, 3780, 3781, 3782], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  5673, 5673, 5673, 5673, 5673, 5673, 5673, 5673, 5673, 5673], [1.1558095238095243, 0.5249999999999997, 0.031690476190477324, -0.08312500000000012, -0.3199285714285625, -0.6768750000000011, 0.05742857142856705, 0.2850000000000009, 0.03835119047618928, 0.08312500000000052  …  1.1334656086978145e-16, 0.011571428571428363, 5.079487178094588e-17, -0.03905357142857214, -0.004881696428571539, 0.004881696428571847, 0.03905357142857165, -5.722419066378492e-16, -5.345116710353537e-16, 1.1965255175550027e-15], 5673, 5673)

## Old and new solvers

Both solvers act on the same assembled `SystemMatrix` and `SystemVector`.


In [3]:
v_old, p_old = solveField(
    K,
    F;
    support=support
)

v_new, p_new = solveField_new(
    K,
    F;
    support=support
)

(
    velocity_abs_error = norm(v_new.a - v_old.a),
    velocity_rel_error = norm(v_new.a - v_old.a) / norm(v_old.a),
    pressure_abs_error = norm(p_new.a - p_old.a),
    pressure_rel_error = norm(p_new.a - p_old.a) / norm(p_old.a)
)


(velocity_abs_error = 7.598619497023073e-11, velocity_rel_error = 3.615607412874076e-11, pressure_abs_error = 6.686658090965639e-9, pressure_rel_error = 1.550878161234118e-9)

## Prepared-system check

This verifies the reduced algebra independently of field reconstruction.


In [4]:
prepared = prepare_multifield_system(
    K,
    F,
    support
)

Xfree = solve_linear_system(
    prepared.A,
    prepared.B;
    solver=:backslash
)

residual = norm(prepared.A * Xfree - prepared.B) / norm(prepared.B)

(
    full_dofs = size(K.A, 1),
    transformed_dofs = size(prepared.T, 2),
    free_dofs = size(prepared.A, 1),
    residual = residual
)


(full_dofs = 5673, transformed_dofs = 4643, free_dofs = 4398, residual = 3.9682376692768e-9)

## Symmetric wrapper

The mixed Stokes matrix is symmetric but indefinite. This deliberately tests the
`:auto_symmetric` path: Cholesky may fail and the solver should then fall back to LU.


In [5]:
Ks = Symmetric(K)

v_sym, p_sym = solveField_new(
    Ks,
    F;
    support=support
)

(
    velocity_rel_error = norm(v_sym.a - v_new.a) / norm(v_new.a),
    pressure_rel_error = norm(p_sym.a - p_new.a) / norm(p_new.a)
)


[ Info: Cholesky factorization failed; falling back to LU.


(velocity_rel_error = 0.0, pressure_rel_error = 0.0)

## Visual check


In [6]:
showDoFResults(v_new, name="v", visible=true)
showDoFResults(p_new, name="p")


1

In [7]:
openPostProcessor()


XOpenIM() failed
Fontconfig warning: using without calling FcInit()


## Multifield + MPC

This section exercises the existing multifield MPC machinery through the new
`prepare → solve → reconstruct` pipeline. Two field-specific constraints are used:

- a velocity MPC tying the right boundary to the `rightbottom` master node;
- a pressure MPC tying `leftbottom` to `rightbottom` while pressure also uses `reducedOrder=true`.

The old and new solvers must produce the same constrained solution.


In [8]:
mpc_v = MPC(
    master="rightbottom",
    slave="right",
    field=V,
    vx=true,
    vy=true
)

mpc_p = MPC(
    master="rightbottom",
    slave="leftbottom",
    field=P,
    p=true
)

mpcs = [mpc_v, mpc_p]


2-element Vector{MultiPointConstraint}:
 MultiPointConstraint("rightbottom", "right", Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 1891, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :v, :fv, false), Dict{Symbol, Bool}(:vy => 1, :vx => 1))
 MultiPointConstraint("rightbottom", "leftbottom", Problem("structured_rect", :ScalarField, 2, 1, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 1891, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :p, :fp, true), Dict{Symbol, Bool}(:p => 1))

In [9]:
v_old_mpc, p_old_mpc = solveField(
    K,
    F;
    support=support,
    mpc=mpcs
)

v_new_mpc, p_new_mpc = solveField_new(
    K,
    F;
    support=support,
    mpc=mpcs
)

(
    velocity_abs_error = norm(v_new_mpc.a - v_old_mpc.a),
    velocity_rel_error = norm(v_new_mpc.a - v_old_mpc.a) / max(norm(v_old_mpc.a), eps()),
    pressure_abs_error = norm(p_new_mpc.a - p_old_mpc.a),
    pressure_rel_error = norm(p_new_mpc.a - p_old_mpc.a) / max(norm(p_old_mpc.a), eps())
)


(velocity_abs_error = 0.0, velocity_rel_error = 0.0, pressure_abs_error = 0.0, pressure_rel_error = 0.0)

### MPC prepared-system check

Besides comparing reconstructed fields, verify the reduced algebra and show how many
transformed DOFs remain after reduced-order interpolation and MPC elimination.


In [10]:
prepared_mpc = prepare_multifield_system(
    K,
    F,
    support;
    mpc=mpcs
)

Xfree_mpc = solve_linear_system(
    prepared_mpc.A,
    prepared_mpc.B;
    solver=:backslash
)

residual_mpc = norm(
    prepared_mpc.A * Xfree_mpc - prepared_mpc.B
) / max(norm(prepared_mpc.B), eps())

(
    full_dofs = size(K.A, 1),
    transformed_dofs = size(prepared_mpc.T, 2),
    free_dofs = size(prepared_mpc.A, 1),
    prunable_dofs = length(prepared_mpc.prunable),
    protected_dofs = length(prepared_mpc.protected),
    residual = residual_mpc
)


(full_dofs = 5673, transformed_dofs = 4582, free_dofs = 4339, prunable_dofs = 4582, protected_dofs = 6, residual = 1.7600972558906974e-11)

### Direct MPC relation check

The reconstructed fields should satisfy the imposed equality constraints directly in
the original full nodal space.


In [11]:
rep_v = LowLevelFEM.mpcRepresentativeMap(V, [mpc_v])
rep_p = LowLevelFEM.mpcRepresentativeMap(P, [mpc_p])

err_v_mpc = maximum(
    abs(v_new_mpc.a[i, 1] - v_new_mpc.a[rep_v[i], 1])
    for i in eachindex(rep_v)
)

err_p_mpc = maximum(
    abs(p_new_mpc.a[i, 1] - p_new_mpc.a[rep_p[i], 1])
    for i in eachindex(rep_p)
)

(
    velocity_mpc_error = err_v_mpc,
    pressure_mpc_error = err_p_mpc
)


(velocity_mpc_error = 0.0, pressure_mpc_error = 0.0)